# Load Battery Dataset

In [1]:
import os
os.chdir('..')
import json
import pandas as pd
import glob

files = glob.glob('data/raw/FastCharge*.json')
print(f"Found {len(files)} files")

Found 140 files


In [2]:
cells = []

for i, file in enumerate(files):
    try:
        print(f"Processing {i}: {file}")
        
        with open(file) as f:
            data = json.load(f)
        
        summary = pd.DataFrame(data['summary'])
        summary = summary[summary['cycle_index'] >= 1]
        
        if len(summary) == 0:
            continue
        
        initial_cap = summary['discharge_capacity'].iloc[0]
        threshold = 0.8 * initial_cap
        below_threshold = summary[summary['discharge_capacity'] < threshold]
        
        if len(below_threshold) > 0:
            eol = below_threshold.iloc[0]['cycle_index']
        else:
            eol = len(summary)
        
        cells.append({
            'cell_id': data['barcode'],
            'EOL': eol,
            'initial_capacity': initial_cap
        })
    except Exception as e:
        print(f"ERROR on {file}: {e}")
        break

print(f"Processed {len(cells)} cells")

Processing 0: data/raw\FastCharge_000000_CH19_structure.json
Processing 1: data/raw\FastCharge_000001_CH16_structure.json
Processing 2: data/raw\FastCharge_000001_CH30_structure.json
Processing 3: data/raw\FastCharge_000001_CH38_structure.json
Processing 4: data/raw\FastCharge_000002_CH10_structure.json
Processing 5: data/raw\FastCharge_000002_CH18_structure.json
Processing 6: data/raw\FastCharge_000002_CH26_structure.json
Processing 7: data/raw\FastCharge_000002_CH2_structure.json
Processing 8: data/raw\FastCharge_000002_CH34_structure.json
Processing 9: data/raw\FastCharge_000002_CH42_structure.json
Processing 10: data/raw\FastCharge_000002_CH47_structure.json
Processing 11: data/raw\FastCharge_000002_CH7_structure.json
Processing 12: data/raw\FastCharge_000003_CH39_structure.json
Processing 13: data/raw\FastCharge_000003_CH40_structure.json
Processing 14: data/raw\FastCharge_000004_CH1_structure.json
Processing 15: data/raw\FastCharge_000004_CH2_structure.json
Processing 16: data/ra

In [3]:
dataset = pd.DataFrame(cells)
dataset.to_csv('data/battery_dataset.csv', index=False)
print("Saved to data/battery_dataset.csv")
print(dataset.head())

Saved to data/battery_dataset.csv
          cell_id   EOL  initial_capacity
0  el150800440551   484          1.045426
1  el150800737229   666          1.062025
2  el150800737366   772          1.067879
3  el150800737234   541          1.051274
4  el150800737329  1009          1.066573
